In [ ]:
import numpy as np
from main import load_cifar_batch  
import matplotlib.pyplot as plt   

In [ ]:
np.random.seed(42)
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]
x_train,y_train = load_cifar_batch('./datasets/cifar-10-batches-py/data_batch_2')
#removed 500 samples of images for are validation   
x_train = x_train[500:]
y_train = y_train[500:]
# from (9500,32,32,3) to (9500,3072)
x_train = x_train.reshape(x_train.shape[0],-1) 
x_norm = x_train/255. 
print(f'x_train after norm the pixels: {x_norm[0]}')
#hot-coded lab
y_coded = np.eye(10)[y_train] 
print(f'one-hot encoded labels for our 9500 images {y_coded}')

#validation sets images -> (500,1)  labels->500
x_val = x_train[:500]
y_val = y_train[:500]
x_val = x_val.reshape(x_val.shape[0],-1) 
val_norm = x_val/255.
print(f'x_val after norm the pixels: {val_norm[0]}')

# Better weight initialization (smaller, from normal distribution)
W = np.random.randn(len(class_names), x_norm.shape[1]) * 0.01
b = np.zeros(shape=(1, 10))



In [ ]:
'''
example for one image 
[pixels....pixels] @ [-1 < w[0] < 1]

'''

def compute_loss(X,W,b,y_tr):
    
    N = X.shape[0]
    
    #raw scores for all the images
    scores = np.dot(X,W.T) + b
    
    #normalized scores with softmax 
    scores -= np.max(scores,axis=1,keepdims=True)
    exp = np.exp(scores) 
    prob = exp / np.sum(exp,axis=1,keepdims=True)
    
    #for each image we have the corresponding label trueY   
    correct_class_prob = prob[np.arange(N),y_tr]
    
    #added 1e-15 to avoid zero inside log 
    losses = -np.log(correct_class_prob + 1e-15)
    
    total_loss = np.sum(losses) / N
    
    return total_loss


compute_loss(x_norm,W,b,y_train)


In [ ]:
def compute_grad(X,W,b,reg,y_true):
    # print(X.shape,W.shape,b.shape)
    N = X.shape[0]
    
    scores = np.dot(X,W.T) + b
    
    scores -= np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    z = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)
    # print(z.shape,y_train.shape)
    
    # Convert integer labels to one-hot
    if y_true.ndim == 1:
        y_one_hot = np.eye(10)[y_true]
    else:
        y_one_hot = y_true
    
    
    # Calculate the error (derivative of loss w.r.t scores) -> log(exp/sum(exp)) -> log(exp(z)) - log(sum(exp(z))) -> z - 1/sum(exp(z)) <- t_label
    dz = z - y_one_hot 
    
    
    dl_dw = np.dot(dz.T,X)
    dl_db = np.sum(dz, axis=0, keepdims=True)
    
    dl_dw/= N
    dl_db/= N
    
    dl_dw += reg * W
    
    return dl_dw,dl_db

compute_grad(x_norm,W,b,reg=1e-5,y_true=y_train)

In [ ]:
def gradient_descent(X, W, b, y_true, 
                     epochs=10,
                     learning_rate=1e-4,
                     batch_size=256
):
    '''
    for each epchos we want to get random indexes for our batches
    
    '''
    hist = []
    N = X.shape[0]
     
    for epoch in range(epochs):
        
        #taking random indices for making our model learn not only from one pattern        
        indices = np.random.permutation(N)
        X_shuffled = X[indices]
        y_shuffled = y_true[indices]
        
        epoch_loss = 0  # Track loss per epoch
        num_batches = 0
        
        for  i in range(0,N,int(batch_size)):
            X_batch = X_shuffled[i:i+int(batch_size)]
            y_batch = y_shuffled[i:i+int(batch_size)]
            
            dl_dw,dl_db = compute_grad(X_batch, W, b, reg=1e-5 ,y_true=y_batch) 
            loss = compute_loss(X_batch, W, b, y_batch)
            
            hist.append(loss)
            epoch_loss += loss 
            num_batches += 1
            
            # print(dl_db.T.shape)
            # print(b.shape)
            
            W -= learning_rate * dl_dw            
            b -= learning_rate * dl_db      
        
        # Print epoch summary
        avg_loss = epoch_loss / num_batches
        print(f'Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}')    

        #print W and b as well to see how they decrease over time 
    
    return W,b,hist
        
        

final_w,final_b ,hist = gradient_descent(x_norm,W,b,y_train,epochs=10,learning_rate=1e-2,batch_size=512)
        

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(hist)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.grid(True)
plt.show()

# Check if loss is decreasing
print(f'First loss: {hist[0]:.4f}')
print(f'Last loss: {hist[-1]:.4f}')
print(f'Improved: {hist[0] - hist[-1]:.4f}')

In [ ]:
# Calculate moving average (smoothed loss)
window_size = 37  # ~1 epoch worth (9500 / 256 ≈ 37 batches)
moving_avg = np.convolve(hist, np.ones(window_size)/window_size, mode='valid')

plt.figure(figsize=(12, 5))
plt.plot(hist, alpha=0.3, label='Actual Loss (Noisy)')
plt.plot(moving_avg, linewidth=2, label='Moving Average (Smoothed)')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Loss: Noisy vs Smoothed')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
pro = np.array([
    [0.1, 0.7, 0.1, 0.1],  # image 0
    [0.8, 0.1, 0.05, 0.05],# image 1
    [0.2, 0.2, 0.5, 0.1]   # image 2
])

y = np.array([1, 0, 2])  # true classes for each image

#        [0,1] -> 0.9
c = pro[np.arange(3), y]
print(c)

In [ ]:
'''
print more variations of graphs and added AdamW for updating learning rate 
and use momentum to not get stuck on while trying to get global min 
'''

In [ ]:
np.random.seed(42)
# add momentum and AdamW for better convergence and faster training
def gradient_descent_mom(X, W, b, y_true, 
                     epochs=10,
                     learning_rate=1e-3,
                     reg = 1e-6,
                     batch_size=512
):
    '''
    for each epchos we want to get random indexes for our batches
    
    '''
    hist = []
    N = X.shape[0]
    rho = 0.92
    vx = np.zeros_like(W)
    
    for epoch in range(epochs):
        
        #taking random indices for making our model learn not only from one pattern        
        indices = np.random.permutation(N)
        X_shuffled = X[indices]
        y_shuffled = y_true[indices]
        
        epoch_loss = 0  # Track loss per epoch
        num_batches = 0

        #update learning rate with cosine annealing
        learning_rate = 0.5 * learning_rate / (1 + np.cos(epoch*np.pi/ N))

        for  i in range(0,N,int(batch_size)):
            X_batch = X_shuffled[i:i+int(batch_size)]
            y_batch = y_shuffled[i:i+int(batch_size)]
            
            dl_dw,dl_db = compute_grad(X_batch, W, b, reg ,y_true=y_batch) 
            loss = compute_loss(X_batch, W, b, y_batch)
            
            hist.append(loss)
            epoch_loss += loss 
            num_batches += 1
        
            vx = rho * vx + dl_dw
            # print(dl_db.T.shape)
            # print(b.shape)
            W -= learning_rate * vx          
            b -= learning_rate * dl_db      
            print(f'W norm: {np.linalg.norm(W):.4f}, b norm: {np.linalg.norm(b):.4f}, Loss: {loss:.4f}')
        # Print epoch summary
        # avg_loss = epoch_loss / num_batches
        # print(f'Epoch {epoch+1}/{epochs} | Avg Loss: {avg_loss:.4f}')    

        #print W and b as well to see how they decrease over time 
    
    return W,b,hist
        
        

w_mom,b_mom ,hist1 = gradient_descent_mom(x_norm,W,b,y_train,epochs=10,learning_rate=1e-3,batch_size=512)
        

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(hist1)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.grid(True)
plt.show()

# Check if loss is decreasing
print(f'First loss: {hist1[0]:.4f}')
print(f'Last loss: {hist1[-1]:.4f}')
print(f'Improved: {hist1[0] - hist1[-1]:.4f}')

In [ ]:
loss_one = compute_loss(x_norm,W,b,y_train)
print(loss_one)
loss_two = compute_loss(x_val,W,b,y_val)
print(loss_two)